In [ ]:
# Make this notebook work from fine-tuning/ or fine-tuning/clinical-rtor/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('clinical-rtor', 'pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 07 (Clinical) · Evaluate the abstractor

"It looks right" doesn't ship in a quality registry. This lab scores the Return-to-OR abstractor two ways: **classification metrics** (precision / recall / F1 on `is_return_to_or`) and an **LLM-as-judge** check that each cited *evidence* string is actually grounded in the source notes.

---
## Step 1 — Config, client, prompt & eval set

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

load_dotenv()

AZURE_OPENAI_ENDPOINT    = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
BASE_MODEL               = os.environ.get('BASE_MODEL', 'gpt-4o-mini-2024-07-18')
BASE_DEPLOYMENT          = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')
SUBSCRIPTION_ID          = os.environ.get('AZURE_SUBSCRIPTION_ID')
RESOURCE_GROUP           = os.environ.get('AZURE_RESOURCE_GROUP')
RESOURCE_NAME            = os.environ.get('AZURE_RESOURCE_NAME')
TENANT_ID                = os.environ.get('AZURE_TENANT_ID')

_cred = DefaultAzureCredential(interactive_browser_tenant_id=TENANT_ID) if TENANT_ID else DefaultAzureCredential()
client = AzureOpenAI(
    azure_endpoint          = AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider = lambda: _cred.get_token('https://cognitiveservices.azure.com/.default').token,
    api_version             = AZURE_OPENAI_API_VERSION,
)
print('client ready ->', AZURE_OPENAI_ENDPOINT)


In [ ]:
import json
from pathlib import Path

CASES = [json.loads(l) for l in Path('data/rtor_cases.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]

RULES_BLOCK = '''
### SPECIFIC ABSTRACTION RULES

Rule 1 - Conflict-resolution order (apply in this EXACT priority; the FIRST match decides):
  1. Planned / staged overrides everything. If the index OR current operative note documents that
     the second procedure was planned, staged, anticipated, or scheduled at the time of the index
     surgery, then is_return_to_or = false (even if it occurs within 30 days).
  2. Unplanned + related + within 30 days = RTOR. If the current surgery is unplanned and treats a
     complication of the index surgery (bleeding, hematoma, surgical-site infection, wound dehiscence,
     anastomotic leak, abscess, graft/flap failure) within 30 days, then is_return_to_or = true.
  3. Unrelated anatomy or new diagnosis = not RTOR (false), regardless of timing.
  4. Outside the 30-day window = not RTOR (false).

Rule 2 - Operating-room requirement. The return must be to an operating room. Bedside, ICU, IR,
  endoscopy-suite, or clinic procedures do NOT count: is_return_to_or = false.

Rule 3 - Evidence requirement. Quote the single most decisive sentence from the source documents
  verbatim, then state which rule it triggers.
'''

SYSTEM_PROMPT = (
    'You are a surgical-quality abstraction assistant for Acme Health. Determine whether the '
    'current operative episode is an unplanned Return to the Operating Room (RTOR) for the index '
    'surgery, applying the rules below.\n'
    + RULES_BLOCK +
    '\n### TASK EXECUTION\n'
    '- Read the provided text thoroughly.\n'
    '- Evaluate the context against the Specific Abstraction Rules, resolving any conflicting data '
    'using the exact order specified in Rule 1.\n'
    '- Output ONLY a valid JSON object with exactly two keys: "is_return_to_or" (boolean) and '
    '"evidence" (string citing the exact text used and how it applies to the rules).\n'
    '- Do not include conversational filler. Do not include markdown formatting like a json fence.'
)

TEMPLATE = '''Patient Timeline:
{patient_timeline_json}

Progress Note Details:
{progress_note_json}

Index Surgery Procedure Description:
{index_surgery_procedure_desc}

Index Surgery Operative Note:
{index_surgery_op_note}

Current Surgery Procedure Description:
{current_surgery_procedure_desc}

Current Surgery Operative Note:
{current_surgery_op_note}

Task: Determine if the current operating note/surgery represents a return to the operating room based strictly on the abstraction rules provided above. Output ONLY the raw JSON object.'''

def build_user_prompt(case):
    return TEMPLATE.format(
        patient_timeline_json        = json.dumps(case.get('patient_timeline', []), indent=2),
        progress_note_json           = json.dumps(case.get('progress_note', {}), indent=2),
        index_surgery_procedure_desc = case.get('index_surgery_procedure_desc', ''),
        index_surgery_op_note        = case.get('index_surgery_op_note', ''),
        current_surgery_procedure_desc = case.get('current_surgery_procedure_desc', ''),
        current_surgery_op_note      = case.get('current_surgery_op_note', ''),
    )

def safe_parse(val):
    '''Safely extract JSON from the LLM response, stripping stray markdown fences.'''
    try:
        clean = str(val).strip()
        if clean.startswith('```'):
            clean = clean.strip('`')
            if clean.startswith('json'):
                clean = clean[4:]
        return json.loads(clean.strip())
    except Exception as e:
        return {'is_return_to_or': None, 'evidence': f'Parse Error: {e} | Raw: {val}'}

print(f'Loaded {len(CASES)} labeled cases. Prompt + parser ready.')
print('--- USER PROMPT for', CASES[0]['case_id'], '(first 500 chars) ---')
print(build_user_prompt(CASES[0])[:500])


In [ ]:
import json
from pathlib import Path
ep = Path('data/rtor_eval.jsonl')
assert ep.exists(), 'Run Lab 00 first to generate data/rtor_eval.jsonl'
EVAL = [json.loads(l) for l in ep.read_text(encoding='utf-8').splitlines() if l.strip()]
print('Eval cases:', len(EVAL))


---
## Step 2 — Run the abstractor over the eval set

In [ ]:
preds = []
for case in EVAL:
    r = client.chat.completions.create(
        model=BASE_DEPLOYMENT,
        messages=[{'role': 'system', 'content': SYSTEM_PROMPT},
                  {'role': 'user',   'content': build_user_prompt(case)}],
        temperature=0.0, max_tokens=300, response_format={'type': 'json_object'},
    )
    p = safe_parse(r.choices[0].message.content)
    preds.append({
        'case_id': case['case_id'],
        'gold': case['gold_is_return_to_or'],
        'pred': p.get('is_return_to_or'),
        'pred_evidence': p.get('evidence'),
        'gold_evidence': case['gold_evidence'],
    })
print('Scored', len(preds), 'cases.')


---
## Step 3 — Classification metrics

In [ ]:
tp = sum(1 for p in preds if p['gold'] and p['pred'] is True)
tn = sum(1 for p in preds if (not p['gold']) and p['pred'] is False)
fp = sum(1 for p in preds if (not p['gold']) and p['pred'] is True)
fn = sum(1 for p in preds if p['gold'] and p['pred'] is False)
unparsed = sum(1 for p in preds if p['pred'] is None)

prec = tp / (tp + fp) if (tp + fp) else 0.0
rec  = tp / (tp + fn) if (tp + fn) else 0.0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
acc  = (tp + tn) / len(preds) if preds else 0.0

print('RETURN-TO-OR CLASSIFICATION SCOREBOARD')
print('=' * 48)
print(f'Accuracy : {acc:.0%}')
print(f'Precision: {prec:.0%}')
print(f'Recall   : {rec:.0%}')
print(f'F1       : {f1:.0%}')
print(f'Confusion: TP={tp}  TN={tn}  FP={fp}  FN={fn}   unparsed={unparsed}')


---
## Step 4 — LLM-as-judge: is the cited evidence grounded?

A correct boolean with a hand-wavy citation still fails an audit. Grade each evidence string 0/1 for quoting a relevant source sentence and naming a plausible rule.

In [ ]:
import json

def judge(p):
    sys = (
        'You grade whether an abstraction citation is well-grounded. Given the GOLD evidence and the '
        'MODEL evidence for a surgical Return-to-OR decision, reply with JSON {"score": 0 or 1, '
        '"reason": "..."}. score=1 means the model quoted a relevant source sentence and named a '
        'plausible rule.'
    )
    r = client.chat.completions.create(
        model=BASE_DEPLOYMENT, temperature=0.0, max_tokens=150,
        response_format={'type': 'json_object'},
        messages=[{'role': 'system', 'content': sys},
                  {'role': 'user',   'content': json.dumps({'gold': p['gold_evidence'], 'model': p['pred_evidence']})}],
    )
    return safe_parse(r.choices[0].message.content).get('score', 0)

scored = [p for p in preds if p['pred'] is not None]
scores = [judge(p) for p in scored]
avg = sum(scores) / len(scores) if scores else 0.0
print(f'Evidence groundedness (LLM-judge): {avg:.0%} over {len(scores)} parsed cases')


---
## Step 5 — One scoreboard per release

In [ ]:
import pandas as pd
board = pd.DataFrame(preds)
board['match'] = board['gold'] == board['pred']
display(board[['case_id', 'gold', 'pred', 'match']])
print(f'\nRelease scoreboard -> classification accuracy {acc:.0%} | evidence groundedness {avg:.0%}')
print('Re-run after Lab 01 fine-tuning to watch both numbers move.')


---
## Takeaways

- Numbers, not vibes: precision/recall on the boolean **and** a groundedness check on the citation.
- The same eval set scores the base model today and the fine-tuned model from Lab 01 — that delta is your release gate.
- Wire this into **Lab 13 (continuous eval)** to catch drift as op-note templates change.